In [2]:
# ============================================
# HEALTHCARE PROJECT - ETL Script
# Step 1: Load, Clean and Store Synthea Data
# ============================================

import pandas as pd
import sqlite3
import os

# --- 1. Define the path to your CSV files ---
# Since Jupyter opened in the same folder as your CSVs, we use "./"
DATA_PATH = "./"

print("Loading CSV files...")

# --- 2. Load each CSV file into a pandas DataFrame ---
patients     = pd.read_csv(os.path.join(DATA_PATH, "patients.csv"))
conditions   = pd.read_csv(os.path.join(DATA_PATH, "conditions.csv"))
encounters   = pd.read_csv(os.path.join(DATA_PATH, "encounters.csv"))
medications  = pd.read_csv(os.path.join(DATA_PATH, "medications.csv"))
procedures   = pd.read_csv(os.path.join(DATA_PATH, "procedures.csv"))
observations = pd.read_csv(os.path.join(DATA_PATH, "observations.csv"))
payers       = pd.read_csv(os.path.join(DATA_PATH, "payers.csv"))
allergies    = pd.read_csv(os.path.join(DATA_PATH, "allergies.csv"))
careplans    = pd.read_csv(os.path.join(DATA_PATH, "careplans.csv"))

print("All files loaded successfully!")

# --- 3. Preview each table ---
print(f"\n📋 Patients:     {patients.shape[0]:,} rows, {patients.shape[1]} columns")
print(f"📋 Conditions:   {conditions.shape[0]:,} rows, {conditions.shape[1]} columns")
print(f"📋 Encounters:   {encounters.shape[0]:,} rows, {encounters.shape[1]} columns")
print(f"📋 Medications:  {medications.shape[0]:,} rows, {medications.shape[1]} columns")
print(f"📋 Procedures:   {procedures.shape[0]:,} rows, {procedures.shape[1]} columns")
print(f"📋 Observations: {observations.shape[0]:,} rows, {observations.shape[1]} columns")
print(f"📋 Payers:       {payers.shape[0]:,} rows, {payers.shape[1]} columns")
print(f"📋 Allergies:    {allergies.shape[0]:,} rows, {allergies.shape[1]} columns")
print(f"📋 Care Plans:   {careplans.shape[0]:,} rows, {careplans.shape[1]} columns")

Loading CSV files...
All files loaded successfully!

📋 Patients:     1,171 rows, 25 columns
📋 Conditions:   8,376 rows, 6 columns
📋 Encounters:   53,346 rows, 15 columns
📋 Medications:  42,989 rows, 13 columns
📋 Procedures:   34,981 rows, 8 columns
📋 Observations: 299,697 rows, 8 columns
📋 Payers:       10 rows, 21 columns
📋 Allergies:    597 rows, 6 columns
📋 Care Plans:   3,483 rows, 9 columns


In [3]:
# ============================================
# STEP 2: Clean Data & Load into SQLite DB
# ============================================

import sqlite3

print(" Starting data cleaning...")

# --- CLEAN PATIENTS ---
# Keep only the columns we need for analysis
patients_clean = patients[[
    'Id', 'BIRTHDATE', 'DEATHDATE', 'GENDER', 'RACE', 'ETHNICITY',
    'CITY', 'STATE', 'COUNTY', 'INCOME', 'MARITAL'
]].copy()

# Calculate age from birthdate
patients_clean['BIRTHDATE'] = pd.to_datetime(patients_clean['BIRTHDATE'])
patients_clean['DEATHDATE'] = pd.to_datetime(patients_clean['DEATHDATE'])
today = pd.Timestamp('2020-01-01')  # Dataset is from 2020
patients_clean['AGE'] = ((today - patients_clean['BIRTHDATE']).dt.days / 365).astype(int)

# Create age groups for population analysis
def age_group(age):
    if age < 18:   return '0-17'
    elif age < 35: return '18-34'
    elif age < 50: return '35-49'
    elif age < 65: return '50-64'
    else:          return '65+'

patients_clean['AGE_GROUP'] = patients_clean['AGE'].apply(age_group)

# Flag deceased patients
patients_clean['IS_DECEASED'] = patients_clean['DEATHDATE'].notna().astype(int)

print(f" Patients cleaned: {len(patients_clean):,} records")

# --- CLEAN CONDITIONS ---
conditions_clean = conditions[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION'
]].copy()
conditions_clean['START'] = pd.to_datetime(conditions_clean['START'])
conditions_clean['STOP']  = pd.to_datetime(conditions_clean['STOP'])

# Flag if condition is chronic (still active / no stop date)
conditions_clean['IS_CHRONIC'] = conditions_clean['STOP'].isna().astype(int)

print(f" Conditions cleaned: {len(conditions_clean):,} records")

# --- CLEAN ENCOUNTERS ---
encounters_clean = encounters[[
    'Id', 'START', 'STOP', 'PATIENT', 'ENCOUNTERCLASS', 'CODE',
    'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST',
    'PAYER_COVERAGE', 'REASONDESCRIPTION'
]].copy()
encounters_clean['START'] = pd.to_datetime(encounters_clean['START'])
encounters_clean['STOP']  = pd.to_datetime(encounters_clean['STOP'])

# Calculate duration of each encounter in hours
encounters_clean['DURATION_HOURS'] = (
    (encounters_clean['STOP'] - encounters_clean['START'])
    .dt.total_seconds() / 3600
).round(2)

# Extract year and month for trend analysis
encounters_clean['YEAR']  = encounters_clean['START'].dt.year
encounters_clean['MONTH'] = encounters_clean['START'].dt.month

print(f"Encounters cleaned: {len(encounters_clean):,} records")

# --- CLEAN MEDICATIONS ---
medications_clean = medications[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE',
    'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE',
    'DISPENSES', 'TOTALCOST', 'REASONDESCRIPTION'
]].copy()
medications_clean['START'] = pd.to_datetime(medications_clean['START'])
medications_clean['STOP']  = pd.to_datetime(medications_clean['STOP'])

# Flag active medications (no stop date)
medications_clean['IS_ACTIVE'] = medications_clean['STOP'].isna().astype(int)

print(f" Medications cleaned: {len(medications_clean):,} records")

# --- CLEAN PROCEDURES ---
procedures_clean = procedures[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER',
    'CODE', 'DESCRIPTION', 'BASE_COST', 'REASONDESCRIPTION'
]].copy()
procedures_clean['START'] = pd.to_datetime(procedures_clean['START'])

print(f" Procedures cleaned: {len(procedures_clean):,} records")

# -----------------------------------------------
# LOAD ALL CLEANED DATA INTO SQLITE DATABASE
# -----------------------------------------------
print("\n Loading into SQLite database...")

# Create the database file in your project folder
conn = sqlite3.connect('healthcare_analytics.db')

# Write each cleaned dataframe as a table in the database
patients_clean.to_sql('patients',    conn, if_exists='replace', index=False)
conditions_clean.to_sql('conditions', conn, if_exists='replace', index=False)
encounters_clean.to_sql('encounters', conn, if_exists='replace', index=False)
medications_clean.to_sql('medications', conn, if_exists='replace', index=False)
procedures_clean.to_sql('procedures', conn, if_exists='replace', index=False)

# Verify tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\n Database created successfully!")
print("📦 Tables in database:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"   → {table[0]}: {count:,} rows")

conn.close()
print("\n ETL Complete! Your database is ready for SQL analysis.")

 Starting data cleaning...


KeyError: "['INCOME'] not in index"

In [4]:
# Check exact column names in patients file
print("PATIENTS columns:")
print(list(patients.columns))

PATIENTS columns:
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE']


In [5]:
# ============================================
# STEP 2: Clean Data & Load into SQLite DB
# ============================================

import sqlite3

print("🔧 Starting data cleaning...")

# --- CLEAN PATIENTS ---
patients_clean = patients[[
    'Id', 'BIRTHDATE', 'DEATHDATE', 'GENDER', 'RACE', 'ETHNICITY',
    'CITY', 'STATE', 'COUNTY', 'MARITAL',
    'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE'
]].copy()

# Calculate age from birthdate
patients_clean['BIRTHDATE'] = pd.to_datetime(patients_clean['BIRTHDATE'])
patients_clean['DEATHDATE'] = pd.to_datetime(patients_clean['DEATHDATE'])
today = pd.Timestamp('2020-01-01')
patients_clean['AGE'] = ((today - patients_clean['BIRTHDATE']).dt.days / 365).astype(int)

# Create age groups
def age_group(age):
    if age < 18:   return '0-17'
    elif age < 35: return '18-34'
    elif age < 50: return '35-49'
    elif age < 65: return '50-64'
    else:          return '65+'

patients_clean['AGE_GROUP'] = patients_clean['AGE'].apply(age_group)

# Flag deceased patients
patients_clean['IS_DECEASED'] = patients_clean['DEATHDATE'].notna().astype(int)

print(f"✅ Patients cleaned: {len(patients_clean):,} records")

# --- CLEAN CONDITIONS ---
conditions_clean = conditions[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION'
]].copy()
conditions_clean['START'] = pd.to_datetime(conditions_clean['START'])
conditions_clean['STOP']  = pd.to_datetime(conditions_clean['STOP'])
conditions_clean['IS_CHRONIC'] = conditions_clean['STOP'].isna().astype(int)

print(f"✅ Conditions cleaned: {len(conditions_clean):,} records")

# --- CLEAN ENCOUNTERS ---
encounters_clean = encounters[[
    'Id', 'START', 'STOP', 'PATIENT', 'ENCOUNTERCLASS', 'CODE',
    'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST',
    'PAYER_COVERAGE', 'REASONDESCRIPTION'
]].copy()
encounters_clean['START'] = pd.to_datetime(encounters_clean['START'])
encounters_clean['STOP']  = pd.to_datetime(encounters_clean['STOP'])
encounters_clean['DURATION_HOURS'] = (
    (encounters_clean['STOP'] - encounters_clean['START'])
    .dt.total_seconds() / 3600
).round(2)
encounters_clean['YEAR']  = encounters_clean['START'].dt.year
encounters_clean['MONTH'] = encounters_clean['START'].dt.month

print(f"✅ Encounters cleaned: {len(encounters_clean):,} records")

# --- CLEAN MEDICATIONS ---
medications_clean = medications[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE',
    'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE',
    'DISPENSES', 'TOTALCOST', 'REASONDESCRIPTION'
]].copy()
medications_clean['START'] = pd.to_datetime(medications_clean['START'])
medications_clean['STOP']  = pd.to_datetime(medications_clean['STOP'])
medications_clean['IS_ACTIVE'] = medications_clean['STOP'].isna().astype(int)

print(f"✅ Medications cleaned: {len(medications_clean):,} records")

# --- CLEAN PROCEDURES ---
procedures_clean = procedures[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER',
    'CODE', 'DESCRIPTION', 'BASE_COST', 'REASONDESCRIPTION'
]].copy()
procedures_clean['START'] = pd.to_datetime(procedures_clean['START'])

print(f"✅ Procedures cleaned: {len(procedures_clean):,} records")

# -----------------------------------------------
# LOAD ALL CLEANED DATA INTO SQLITE DATABASE
# -----------------------------------------------
print("\n💾 Loading into SQLite database...")

conn = sqlite3.connect('healthcare_analytics.db')

patients_clean.to_sql('patients',     conn, if_exists='replace', index=False)
conditions_clean.to_sql('conditions', conn, if_exists='replace', index=False)
encounters_clean.to_sql('encounters', conn, if_exists='replace', index=False)
medications_clean.to_sql('medications', conn, if_exists='replace', index=False)
procedures_clean.to_sql('procedures', conn, if_exists='replace', index=False)

# Verify tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\n✅ Database created successfully!")
print("📦 Tables in database:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"   → {table[0]}: {count:,} rows")

conn.close()
print("\n🎉 ETL Complete! Your database is ready for SQL analysis.")

🔧 Starting data cleaning...
✅ Patients cleaned: 1,171 records
✅ Conditions cleaned: 8,376 records
✅ Encounters cleaned: 53,346 records
✅ Medications cleaned: 42,989 records


KeyError: "['START', 'STOP'] not in index"

In [6]:
# Check exact column names in procedures file
print("PROCEDURES columns:")
print(list(procedures.columns))

PROCEDURES columns:
['DATE', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST', 'REASONCODE', 'REASONDESCRIPTION']


In [7]:
# ============================================
# STEP 2 (FIXED): Clean Procedures & Load All
# ============================================

import sqlite3

print("🔧 Starting data cleaning...")

# --- CLEAN PATIENTS ---
patients_clean = patients[[
    'Id', 'BIRTHDATE', 'DEATHDATE', 'GENDER', 'RACE', 'ETHNICITY',
    'CITY', 'STATE', 'COUNTY', 'MARITAL',
    'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE'
]].copy()

patients_clean['BIRTHDATE'] = pd.to_datetime(patients_clean['BIRTHDATE'])
patients_clean['DEATHDATE'] = pd.to_datetime(patients_clean['DEATHDATE'])
today = pd.Timestamp('2020-01-01')
patients_clean['AGE'] = ((today - patients_clean['BIRTHDATE']).dt.days / 365).astype(int)

def age_group(age):
    if age < 18:   return '0-17'
    elif age < 35: return '18-34'
    elif age < 50: return '35-49'
    elif age < 65: return '50-64'
    else:          return '65+'

patients_clean['AGE_GROUP'] = patients_clean['AGE'].apply(age_group)
patients_clean['IS_DECEASED'] = patients_clean['DEATHDATE'].notna().astype(int)
print(f"✅ Patients cleaned:    {len(patients_clean):,} records")

# --- CLEAN CONDITIONS ---
conditions_clean = conditions[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION'
]].copy()
conditions_clean['START'] = pd.to_datetime(conditions_clean['START'])
conditions_clean['STOP']  = pd.to_datetime(conditions_clean['STOP'])
conditions_clean['IS_CHRONIC'] = conditions_clean['STOP'].isna().astype(int)
print(f"✅ Conditions cleaned:  {len(conditions_clean):,} records")

# --- CLEAN ENCOUNTERS ---
encounters_clean = encounters[[
    'Id', 'START', 'STOP', 'PATIENT', 'ENCOUNTERCLASS', 'CODE',
    'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST',
    'PAYER_COVERAGE', 'REASONDESCRIPTION'
]].copy()
encounters_clean['START'] = pd.to_datetime(encounters_clean['START'])
encounters_clean['STOP']  = pd.to_datetime(encounters_clean['STOP'])
encounters_clean['DURATION_HOURS'] = (
    (encounters_clean['STOP'] - encounters_clean['START'])
    .dt.total_seconds() / 3600
).round(2)
encounters_clean['YEAR']  = encounters_clean['START'].dt.year
encounters_clean['MONTH'] = encounters_clean['START'].dt.month
print(f"✅ Encounters cleaned:  {len(encounters_clean):,} records")

# --- CLEAN MEDICATIONS ---
medications_clean = medications[[
    'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE',
    'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE',
    'DISPENSES', 'TOTALCOST', 'REASONDESCRIPTION'
]].copy()
medications_clean['START'] = pd.to_datetime(medications_clean['START'])
medications_clean['STOP']  = pd.to_datetime(medications_clean['STOP'])
medications_clean['IS_ACTIVE'] = medications_clean['STOP'].isna().astype(int)
print(f"✅ Medications cleaned: {len(medications_clean):,} records")

# --- CLEAN PROCEDURES (FIXED - uses DATE not START/STOP) ---
procedures_clean = procedures[[
    'DATE', 'PATIENT', 'ENCOUNTER',
    'CODE', 'DESCRIPTION', 'BASE_COST', 'REASONDESCRIPTION'
]].copy()
procedures_clean['DATE'] = pd.to_datetime(procedures_clean['DATE'])
procedures_clean['YEAR']  = procedures_clean['DATE'].dt.year
procedures_clean['MONTH'] = procedures_clean['DATE'].dt.month
print(f"✅ Procedures cleaned:  {len(procedures_clean):,} records")

# -----------------------------------------------
# LOAD ALL CLEANED DATA INTO SQLITE DATABASE
# -----------------------------------------------
print("\n💾 Loading into SQLite database...")

conn = sqlite3.connect('healthcare_analytics.db')

patients_clean.to_sql('patients',       conn, if_exists='replace', index=False)
conditions_clean.to_sql('conditions',   conn, if_exists='replace', index=False)
encounters_clean.to_sql('encounters',   conn, if_exists='replace', index=False)
medications_clean.to_sql('medications', conn, if_exists='replace', index=False)
procedures_clean.to_sql('procedures',   conn, if_exists='replace', index=False)

# Verify tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\n✅ Database created successfully!")
print("📦 Tables in database:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"   → {table[0]}: {count:,} rows")

conn.close()
print("\n🎉 ETL Complete! Your database is ready for SQL analysis.")


🔧 Starting data cleaning...
✅ Patients cleaned:    1,171 records
✅ Conditions cleaned:  8,376 records
✅ Encounters cleaned:  53,346 records
✅ Medications cleaned: 42,989 records
✅ Procedures cleaned:  34,981 records

💾 Loading into SQLite database...

✅ Database created successfully!
📦 Tables in database:
   → patients: 1,171 rows
   → conditions: 8,376 rows
   → encounters: 53,346 rows
   → medications: 42,989 rows
   → procedures: 34,981 rows

🎉 ETL Complete! Your database is ready for SQL analysis.


In [8]:
# ============================================
# STEP 3: SQL Analysis Queries
# ============================================

import sqlite3
import pandas as pd

# Connect to our database
conn = sqlite3.connect('healthcare_analytics.db')

print("=" * 55)
print("📊 HEALTHCARE POPULATION ANALYSIS")
print("=" * 55)

# ---------------------------------------------------
# QUERY 1: Population Overview by Gender
# ---------------------------------------------------
q1 = pd.read_sql_query("""
    SELECT 
        GENDER,
        COUNT(*) as total_patients,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM patients), 1) as percentage,
        ROUND(AVG(AGE), 1) as avg_age,
        SUM(IS_DECEASED) as deceased,
        ROUND(AVG(HEALTHCARE_EXPENSES), 2) as avg_healthcare_expenses
    FROM patients
    GROUP BY GENDER
    ORDER BY total_patients DESC
""", conn)
print("\n🔹 QUERY 1: Population by Gender")
print(q1.to_string(index=False))

# ---------------------------------------------------
# QUERY 2: Population by Age Group
# ---------------------------------------------------
q2 = pd.read_sql_query("""
    SELECT 
        AGE_GROUP,
        COUNT(*) as total_patients,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM patients), 1) as percentage,
        ROUND(AVG(HEALTHCARE_EXPENSES), 2) as avg_expenses,
        ROUND(AVG(HEALTHCARE_COVERAGE), 2) as avg_coverage
    FROM patients
    GROUP BY AGE_GROUP
    ORDER BY AGE_GROUP
""", conn)
print("\n🔹 QUERY 2: Population by Age Group")
print(q2.to_string(index=False))

# ---------------------------------------------------
# QUERY 3: Top 10 Most Common Conditions
# ---------------------------------------------------
q3 = pd.read_sql_query("""
    SELECT 
        DESCRIPTION as condition,
        COUNT(*) as total_cases,
        SUM(IS_CHRONIC) as chronic_cases,
        ROUND(SUM(IS_CHRONIC) * 100.0 / COUNT(*), 1) as pct_chronic,
        COUNT(DISTINCT PATIENT) as patients_affected
    FROM conditions
    GROUP BY DESCRIPTION
    ORDER BY total_cases DESC
    LIMIT 10
""", conn)
print("\n🔹 QUERY 3: Top 10 Most Common Conditions")
print(q3.to_string(index=False))

# ---------------------------------------------------
# QUERY 4: Average Cost by Encounter Type
# ---------------------------------------------------
q4 = pd.read_sql_query("""
    SELECT 
        ENCOUNTERCLASS as encounter_type,
        COUNT(*) as total_encounters,
        ROUND(AVG(TOTAL_CLAIM_COST), 2) as avg_cost,
        ROUND(MIN(TOTAL_CLAIM_COST), 2) as min_cost,
        ROUND(MAX(TOTAL_CLAIM_COST), 2) as max_cost,
        ROUND(SUM(TOTAL_CLAIM_COST), 2) as total_cost
    FROM encounters
    GROUP BY ENCOUNTERCLASS
    ORDER BY total_encounters DESC
""", conn)
print("\n🔹 QUERY 4: Cost by Encounter Type")
print(q4.to_string(index=False))

# ---------------------------------------------------
# QUERY 5: High Risk Patients (3+ Conditions)
# ---------------------------------------------------
q5 = pd.read_sql_query("""
    SELECT 
        p.Id as patient_id,
        p.GENDER,
        p.AGE,
        p.AGE_GROUP,
        p.RACE,
        COUNT(DISTINCT c.DESCRIPTION) as num_conditions,
        ROUND(p.HEALTHCARE_EXPENSES, 2) as total_expenses
    FROM patients p
    JOIN conditions c ON p.Id = c.PATIENT
    GROUP BY p.Id
    HAVING num_conditions >= 3
    ORDER BY num_conditions DESC, total_expenses DESC
    LIMIT 15
""", conn)
print("\n🔹 QUERY 5: High Risk Patients (3+ Conditions)")
print(q5.to_string(index=False))

# ---------------------------------------------------
# QUERY 6: Top 10 Most Prescribed Medications
# ---------------------------------------------------
q6 = pd.read_sql_query("""
    SELECT 
        DESCRIPTION as medication,
        COUNT(*) as times_prescribed,
        COUNT(DISTINCT PATIENT) as unique_patients,
        ROUND(AVG(TOTALCOST), 2) as avg_cost,
        ROUND(SUM(TOTALCOST), 2) as total_cost,
        SUM(IS_ACTIVE) as currently_active
    FROM medications
    GROUP BY DESCRIPTION
    ORDER BY times_prescribed DESC
    LIMIT 10
""", conn)
print("\n🔹 QUERY 6: Top 10 Most Prescribed Medications")
print(q6.to_string(index=False))

# ---------------------------------------------------
# QUERY 7: 30-Day Readmission Rate
# ---------------------------------------------------
q7 = pd.read_sql_query("""
    SELECT
        e1.PATIENT,
        e1.START as first_visit,
        e2.START as readmission,
        ROUND(JULIANDAY(e2.START) - JULIANDAY(e1.START), 0) as days_between,
        e1.ENCOUNTERCLASS as first_encounter_type,
        e1.REASONDESCRIPTION as reason
    FROM encounters e1
    JOIN encounters e2 
        ON e1.PATIENT = e2.PATIENT
        AND e2.START > e1.START
        AND JULIANDAY(e2.START) - JULIANDAY(e1.START) <= 30
    WHERE e1.ENCOUNTERCLASS = 'inpatient'
    ORDER BY days_between ASC
    LIMIT 15
""", conn)
print("\n🔹 QUERY 7: 30-Day Readmissions (Inpatient)")
print(q7.to_string(index=False))

# ---------------------------------------------------
# QUERY 8: Healthcare Costs by Race & Ethnicity
# ---------------------------------------------------
q8 = pd.read_sql_query("""
    SELECT 
        RACE,
        ETHNICITY,
        COUNT(*) as total_patients,
        ROUND(AVG(HEALTHCARE_EXPENSES), 2) as avg_expenses,
        ROUND(AVG(HEALTHCARE_COVERAGE), 2) as avg_coverage,
        ROUND(AVG(HEALTHCARE_EXPENSES - HEALTHCARE_COVERAGE), 2) as avg_out_of_pocket
    FROM patients
    GROUP BY RACE, ETHNICITY
    ORDER BY avg_expenses DESC
""", conn)
print("\n🔹 QUERY 8: Healthcare Costs by Race & Ethnicity")
print(q8.to_string(index=False))

conn.close()
print("\n" + "=" * 55)
print("✅ All queries complete!")
print("=" * 55)

📊 HEALTHCARE POPULATION ANALYSIS

🔹 QUERY 1: Population by Gender
GENDER  total_patients  percentage  avg_age  deceased  avg_healthcare_expenses
     F             609        52.0     42.6        69                754166.78
     M             562        48.0     45.3       102                776616.07

🔹 QUERY 2: Population by Age Group
AGE_GROUP  total_patients  percentage  avg_expenses  avg_coverage
     0-17             230        19.6     186990.75       2501.54
    18-34             245        20.9     509606.13       5372.89
    35-49             209        17.8     793010.94       8760.74
    50-64             210        17.9    1057091.99      14067.52
      65+             277        23.7    1227999.33      30568.05

🔹 QUERY 3: Top 10 Most Common Conditions
                              condition  total_cases  chronic_cases  pct_chronic  patients_affected
             Viral sinusitis (disorder)         1248              6          0.5                743
     Acute viral pharyn

In [9]:
# ============================================
# STEP 4: Export Analysis Results for Tableau
# ============================================

import sqlite3
import pandas as pd
import os

# Create a folder to store all Tableau-ready files
os.makedirs('tableau_data', exist_ok=True)

conn = sqlite3.connect('healthcare_analytics.db')

print("📤 Exporting data for Tableau...\n")

# ---------------------------------------------------
# EXPORT 1: Population Overview
# ---------------------------------------------------
pop_gender = pd.read_sql_query("""
    SELECT 
        GENDER,
        AGE_GROUP,
        RACE,
        ETHNICITY,
        MARITAL,
        STATE,
        COUNTY,
        AGE,
        IS_DECEASED,
        HEALTHCARE_EXPENSES,
        HEALTHCARE_COVERAGE,
        ROUND(HEALTHCARE_EXPENSES - HEALTHCARE_COVERAGE, 2) as OUT_OF_POCKET
    FROM patients
""", conn)
pop_gender.to_csv('tableau_data/01_patients_overview.csv', index=False)
print(f"Exported: 01_patients_overview.csv  ({len(pop_gender):,} rows)")

# ---------------------------------------------------
# EXPORT 2: Conditions Summary
# ---------------------------------------------------
conditions_summary = pd.read_sql_query("""
    SELECT 
        c.DESCRIPTION as condition,
        c.IS_CHRONIC,
        c.START,
        c.STOP,
        p.GENDER,
        p.AGE_GROUP,
        p.RACE,
        p.ETHNICITY,
        p.HEALTHCARE_EXPENSES
    FROM conditions c
    JOIN patients p ON c.PATIENT = p.Id
""", conn)
conditions_summary.to_csv('tableau_data/02_conditions_summary.csv', index=False)
print(f"Exported: 02_conditions_summary.csv ({len(conditions_summary):,} rows)")

# ---------------------------------------------------
# EXPORT 3: Encounters & Costs
# ---------------------------------------------------
encounters_cost = pd.read_sql_query("""
    SELECT 
        e.ENCOUNTERCLASS as encounter_type,
        e.DESCRIPTION as encounter_description,
        e.REASONDESCRIPTION as

_IncompleteInputError: incomplete input (2579786721.py, line 61)

In [10]:
# ============================================
# STEP 4: Export Analysis Results for Tableau
# ============================================

import sqlite3
import pandas as pd
import os

# Create a folder to store all Tableau-ready files
os.makedirs('tableau_data', exist_ok=True)

conn = sqlite3.connect('healthcare_analytics.db')

print("📤 Exporting data for Tableau...\n")

# ---------------------------------------------------
# EXPORT 1: Population Overview
# ---------------------------------------------------
pop_gender = pd.read_sql_query("""
    SELECT 
        GENDER,
        AGE_GROUP,
        RACE,
        ETHNICITY,
        MARITAL,
        STATE,
        COUNTY,
        AGE,
        IS_DECEASED,
        HEALTHCARE_EXPENSES,
        HEALTHCARE_COVERAGE,
        ROUND(HEALTHCARE_EXPENSES - HEALTHCARE_COVERAGE, 2) as OUT_OF_POCKET
    FROM patients
""", conn)
pop_gender.to_csv('tableau_data/01_patients_overview.csv', index=False)
print(f"✅ Exported: 01_patients_overview.csv  ({len(pop_gender):,} rows)")

# ---------------------------------------------------
# EXPORT 2: Conditions Summary
# ---------------------------------------------------
conditions_summary = pd.read_sql_query("""
    SELECT 
        c.DESCRIPTION as condition,
        c.IS_CHRONIC,
        c.START,
        c.STOP,
        p.GENDER,
        p.AGE_GROUP,
        p.RACE,
        p.ETHNICITY,
        p.HEALTHCARE_EXPENSES
    FROM conditions c
    JOIN patients p ON c.PATIENT = p.Id
""", conn)
conditions_summary.to_csv('tableau_data/02_conditions_summary.csv', index=False)
print(f"✅ Exported: 02_conditions_summary.csv ({len(conditions_summary):,} rows)")

# ---------------------------------------------------
# EXPORT 3: Encounters & Costs
# ---------------------------------------------------
encounters_cost = pd.read_sql_query("""
    SELECT 
        e.ENCOUNTERCLASS as encounter_type,
        e.DESCRIPTION as encounter_description,
        e.REASONDESCRIPTION as reason,
        e.TOTAL_CLAIM_COST,
        e.BASE_ENCOUNTER_COST,
        e.PAYER_COVERAGE,
        ROUND(e.TOTAL_CLAIM_COST - e.PAYER_COVERAGE, 2) as PATIENT_COST,
        e.DURATION_HOURS,
        e.YEAR,
        e.MONTH,
        p.GENDER,
        p.AGE_GROUP,
        p.RACE,
        p.ETHNICITY
    FROM encounters e
    JOIN patients p ON e.PATIENT = p.Id
""", conn)
encounters_cost.to_csv('tableau_data/03_encounters_cost.csv', index=False)
print(f"✅ Exported: 03_encounters_cost.csv    ({len(encounters_cost):,} rows)")

# ---------------------------------------------------
# EXPORT 4: Medications Summary
# ---------------------------------------------------
medications_summary = pd.read_sql_query("""
    SELECT 
        m.DESCRIPTION as medication,
        m.REASONDESCRIPTION as prescribed_for,
        m.TOTALCOST as total_cost,
        m.BASE_COST as base_cost,
        m.DISPENSES as dispenses,
        m.IS_ACTIVE,
        m.START,
        p.GENDER,
        p.AGE_GROUP,
        p.RACE,
        p.ETHNICITY
    FROM medications m
    JOIN patients p ON m.PATIENT = p.Id
""", conn)
medications_summary.to_csv('tableau_data/04_medications_summary.csv', index=False)
print(f"✅ Exported: 04_medications_summary.csv ({len(medications_summary):,} rows)")

# ---------------------------------------------------
# EXPORT 5: High Risk Patients
# ---------------------------------------------------
high_risk = pd.read_sql_query("""
    SELECT 
        p.Id as patient_id,
        p.GENDER,
        p.AGE,
        p.AGE_GROUP,
        p.RACE,
        p.ETHNICITY,
        p.COUNTY,
        p.HEALTHCARE_EXPENSES,
        p.HEALTHCARE_COVERAGE,
        ROUND(p.HEALTHCARE_EXPENSES - p.HEALTHCARE_COVERAGE, 2) as OUT_OF_POCKET,
        COUNT(DISTINCT c.DESCRIPTION) as num_conditions,
        GROUP_CONCAT(DISTINCT c.DESCRIPTION) as conditions_list
    FROM patients p
    JOIN conditions c ON p.Id = c.PATIENT
    GROUP BY p.Id
    ORDER BY num_conditions DESC
""", conn)
high_risk.to_csv('tableau_data/05_high_risk_patients.csv', index=False)
print(f"✅ Exported: 05_high_risk_patients.csv  ({len(high_risk):,} rows)")

# ---------------------------------------------------
# EXPORT 6: Readmissions
# ---------------------------------------------------
readmissions = pd.read_sql_query("""
    SELECT
        e1.PATIENT,
        e1.START as first_visit,
        e2.START as readmission_date,
        ROUND(JULIANDAY(e2.START) - JULIANDAY(e1.START), 0) as days_to_readmission,
        e1.ENCOUNTERCLASS as encounter_type,
        e1.REASONDESCRIPTION as reason,
        e1.TOTAL_CLAIM_COST as first_visit_cost,
        e2.TOTAL_CLAIM_COST as readmission_cost,
        p.GENDER,
        p.AGE_GROUP,
        p.RACE
    FROM encounters e1
    JOIN encounters e2 
        ON e1.PATIENT = e2.PATIENT
        AND e2.START > e1.START
        AND JULIANDAY(e2.START) - JULIANDAY(e1.START) <= 30
    JOIN patients p ON e1.PATIENT = p.Id
    WHERE e1.ENCOUNTERCLASS = 'inpatient'
""", conn)
readmissions.to_csv('tableau_data/06_readmissions.csv', index=False)
print(f"✅ Exported: 06_readmissions.csv        ({len(readmissions):,} rows)")

# ---------------------------------------------------
# EXPORT 7: Procedures Summary
# ---------------------------------------------------
procedures_summary = pd.read_sql_query("""
    SELECT 
        pr.DESCRIPTION as procedure_name,
        pr.REASONDESCRIPTION as reason,
        pr.BASE_COST as cost,
        pr.YEAR,
        pr.MONTH,
        p.GENDER,
        p.AGE_GROUP,
        p.RACE,
        p.ETHNICITY
    FROM procedures pr
    JOIN patients p ON pr.PATIENT = p.Id
""", conn)
procedures_summary.to_csv('tableau_data/07_procedures_summary.csv', index=False)
print(f"✅ Exported: 07_procedures_summary.csv  ({len(procedures_summary):,} rows)")

conn.close()

print("\n" + "=" * 50)
print("🎉 All files exported to 'tableau_data' folder!")
print("=" * 50)
print("\n📁 Your Tableau files are ready:")
for f in os.listdir('tableau_data'):
    size = os.path.getsize(f'tableau_data/{f}') / 1024
    print(f"   → {f} ({size:.1f} KB)")

📤 Exporting data for Tableau...

✅ Exported: 01_patients_overview.csv  (1,171 rows)
✅ Exported: 02_conditions_summary.csv (8,376 rows)
✅ Exported: 03_encounters_cost.csv    (53,346 rows)
✅ Exported: 04_medications_summary.csv (42,989 rows)
✅ Exported: 05_high_risk_patients.csv  (1,152 rows)
✅ Exported: 06_readmissions.csv        (3,838 rows)
✅ Exported: 07_procedures_summary.csv  (34,981 rows)

🎉 All files exported to 'tableau_data' folder!

📁 Your Tableau files are ready:
   → 05_high_risk_patients.csv (315.0 KB)
   → 01_patients_overview.csv (108.1 KB)
   → 03_encounters_cost.csv (5762.2 KB)
   → 06_readmissions.csv (507.5 KB)
   → 02_conditions_summary.csv (775.7 KB)
   → 07_procedures_summary.csv (2895.2 KB)
   → 04_medications_summary.csv (5602.7 KB)
